In [ ]:
import json, os, subprocess, sys, time, shutil, zipfile
from pathlib import Path
def run(cmd, cwd=None):
    print("$", cmd, flush=True); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end="", flush=True)
    rc = p.wait(); print(f"[exit {rc} after {time.time()-t:.0f}s]", flush=True); return rc
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
IN = Path("/kaggle/input/dermdoc-lipsync-in")
if not (IN / "jobs.json").exists():
    IN = next(Path("/kaggle/input").rglob("jobs.json")).parent
jobs = json.loads((IN / "jobs.json").read_text())
W = Path("/kaggle/working"); OUT = Path("/tmp/ls_out"); OUT.mkdir(exist_ok=True)
run("git clone -q https://github.com/bytedance/LatentSync.git /tmp/LatentSync")
for pk in ["diffusers==0.32.2", "transformers==4.48.0", "huggingface-hub==0.30.2", "accelerate==1.2.1", "peft", "einops", "omegaconf", "opencv-python", "mediapipe",
           "python_speech_features", "librosa", "scenedetect", "ffmpeg-python", "imageio", "imageio-ffmpeg", "lpips",
           "face-alignment", "kornia", "insightface", "onnxruntime-gpu", "DeepCache"]:
    run(f"pip install -q {pk}")
if run("pip install -q eva-decord") != 0:
    run("pip install -q decord")
run("python -c 'import decord, mediapipe, diffusers, insightface; print(\"imports ok\")'")
from huggingface_hub import snapshot_download
def fetch(repo):
    shutil.rmtree("/tmp/LatentSync/checkpoints", ignore_errors=True)
    snapshot_download(repo, local_dir="/tmp/LatentSync/checkpoints")
    print(sorted(str(x) for x in Path("/tmp/LatentSync/checkpoints").rglob("*") if x.is_file()), flush=True)
VER = "ByteDance/LatentSync-1.6"
cfg = "configs/unet/stage2_512.yaml"
fetch(VER)
def infer(j, cfg, steps=20):
    return run(f"python -m scripts.inference --unet_config_path {cfg} --inference_ckpt_path checkpoints/latentsync_unet.pt "
               f"--inference_steps {steps} --guidance_scale 1.5 --enable_deepcache --video_path {IN/j['video']} "
               f"--audio_path {IN/j['audio']} --video_out_path {OUT/j['out']}", cwd="/tmp/LatentSync")
ok = 0
for i, j in enumerate(jobs):
    if (OUT / j["out"]).exists(): continue
    rc = infer(j, cfg)
    if rc != 0 and cfg.endswith("512.yaml"):   # 1.6 (512px) did not fit / failed -> 1.5 (256px)
        print("!! falling back to LatentSync-1.5", flush=True)
        fetch("ByteDance/LatentSync-1.5")
        cfg = "configs/unet/stage2.yaml"; rc = infer(j, cfg)
    ok += rc == 0
    print(f"== {i+1}/{len(jobs)} {j['out']}: {'OK' if rc == 0 else 'FAILED'} ({cfg})", flush=True)
with zipfile.ZipFile(W / "lipsync_out.zip", "w") as zf:
    for f in OUT.glob("*.mp4"): zf.write(f, f.name)
    zf.writestr("model.txt", cfg)
print(f"DONE {ok}/{len(jobs)}")